In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS mvp.gold
COMMENT 'Camada Gold: modelo dimensional Star Schema e visões executivas';

# Schema e Dimensões

In [0]:
%sql
-- ==============================================================================
-- 1. INFRAESTRUTURA DO SCHEMA GOLD
-- ==============================================================================
CREATE SCHEMA IF NOT EXISTS mvp.gold
COMMENT 'Camada Gold: modelo dimensional Star Schema e visões executivas de negócio';

-- ==============================================================================
-- 2. DIM_TEMPO (Competências Mensais)
-- ==============================================================================
CREATE TABLE IF NOT EXISTS mvp.gold.dim_tempo (
    sk_tempo            INT,
    data_referencia     DATE,
    ano                 INT,
    mes                 INT,
    ano_mes_desc        STRING,
    trimestre           INT,
    semestre            INT,
    nome_mes            STRING
)
COMMENT 'Dimensão de Tempo Mensal das transações e contestações';

MERGE INTO mvp.gold.dim_tempo AS target
USING (
    SELECT DISTINCT
        anomes                                                    AS sk_tempo,
        data_competencia                                          AS data_referencia,
        ano,
        mes,
        CONCAT(CAST(ano AS STRING), '/', LPAD(CAST(mes AS STRING), 2, '0')) AS ano_mes_desc,
        QUARTER(data_competencia)                                 AS trimestre,
        CASE WHEN mes <= 6 THEN 1 ELSE 2 END                      AS semestre,
        DATE_FORMAT(data_competencia, 'MMMM')                     AS nome_mes
    FROM mvp.silver.pix_transacoes
) AS source
ON target.sk_tempo = source.sk_tempo
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *;

-- ==============================================================================
-- 3. DIM_REGIAO (Conformada com IBGE)
-- ==============================================================================
CREATE TABLE IF NOT EXISTS mvp.gold.dim_regiao (
    sk_regiao           BIGINT,
    id_regiao_ibge      INT,
    nome_regiao         STRING,
    populacao_recente   BIGINT
)
COMMENT 'Dimensão Geográfica das Grandes Regiões Brasileiras com População IBGE';

MERGE INTO mvp.gold.dim_regiao AS target
USING (
    SELECT 
        xxhash64(nome_regiao) AS sk_regiao,
        id_regiao_ibge,
        nome_regiao,
        populacao_residente   AS populacao_recente
    FROM mvp.silver.ibge_populacao
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY nome_regiao 
        ORDER BY ano_referencia DESC
    ) = 1
    
    UNION ALL
    
    -- Registro canônico para registros sem região identificada
    SELECT 
        xxhash64('NÃO INFORMADO') AS sk_regiao,
        -1                        AS id_regiao_ibge,
        'NÃO INFORMADO'           AS nome_regiao,
        0                         AS populacao_recente
) AS source
ON target.sk_regiao = source.sk_regiao
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *;

-- ==============================================================================
-- 4. DIM_PERFIL_CLIENTE (Tipo de Pessoa, Faixa Etária e Ordem Visual)
-- ==============================================================================
CREATE OR REPLACE TABLE mvp.gold.dim_perfil_cliente (
    sk_perfil           BIGINT,
    tipo_pessoa         STRING,
    faixa_etaria        STRING,
    ordem_faixa_etaria  INT
)
USING DELTA
COMMENT 'Dimensão de Perfil demográfico e cadastral com ordenação etária';

-- Popula os dados com a ordenação incluída
MERGE INTO mvp.gold.dim_perfil_cliente AS target
USING (
    SELECT DISTINCT
        xxhash64(tipo_pessoa, faixa_etaria) AS sk_perfil,
        tipo_pessoa,
        faixa_etaria,
        CASE 
            WHEN faixa_etaria LIKE '%<20%' OR faixa_etaria LIKE '%Até 19%' THEN 1
            WHEN faixa_etaria LIKE '%20%29%' THEN 2
            WHEN faixa_etaria LIKE '%30%39%' THEN 3
            WHEN faixa_etaria LIKE '%40%49%' THEN 4
            WHEN faixa_etaria LIKE '%50%59%' THEN 5
            WHEN faixa_etaria LIKE '%60%' THEN 6
            ELSE 9
        END AS ordem_faixa_etaria
    FROM (
        SELECT pag_pfpj AS tipo_pessoa, pag_faixa_etaria AS faixa_etaria FROM mvp.silver.pix_transacoes
        UNION
        SELECT rec_pfpj AS tipo_pessoa, rec_faixa_etaria AS faixa_etaria FROM mvp.silver.pix_transacoes
    )
) AS source
ON target.sk_perfil = source.sk_perfil
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *;

-- ==============================================================================
-- 5. DIM_MODALIDADE_PIX (Iniciação com Legenda Bacen, Ordem, Natureza e Finalidade)
-- ==============================================================================
CREATE OR REPLACE TABLE mvp.gold.dim_modalidade_pix (
    sk_modalidade           BIGINT,
    forma_iniciacao         STRING,
    forma_iniciacao_desc    STRING,
    ordem_iniciacao         INT,
    natureza                STRING,
    finalidade              STRING
)
USING DELTA
COMMENT 'Dimensão das características operacionais da transação Pix com semântica regulatória';

MERGE INTO mvp.gold.dim_modalidade_pix AS target
USING (
    SELECT DISTINCT
        xxhash64(forma_iniciacao, natureza, finalidade) AS sk_modalidade,
        forma_iniciacao,
        forma_iniciacao_desc,
        ordem_iniciacao,
        natureza,
        finalidade
    FROM mvp.silver.pix_transacoes
) AS source
ON target.sk_modalidade = source.sk_modalidade
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
99,0,0,99


Formatações padrão brasileiro

In [0]:
%sql
-- 1. Formatação de Moeda Brasileira (R$ 1.234.567,89)
CREATE OR REPLACE FUNCTION mvp.gold.fmt_moeda_br(valor DECIMAL(18, 2))
RETURNS STRING
LANGUAGE SQL
DETERMINISTIC
CONTAINS SQL
COMMENT 'Converte número decimal para padrão monetário brasileiro R$ 1.234,56'
RETURN CASE 
    WHEN valor IS NULL THEN NULL
    ELSE CONCAT('R$ ', TRANSLATE(FORMAT_NUMBER(valor, 2), ',.', '.,'))
END;

-- 2. Formatação de Números Inteiros (1.234.567)
CREATE OR REPLACE FUNCTION mvp.gold.fmt_inteiro_br(valor BIGINT)
RETURNS STRING
LANGUAGE SQL
DETERMINISTIC
CONTAINS SQL
COMMENT 'Converte número inteiro para formato brasileiro com separador de milhar'
RETURN CASE 
    WHEN valor IS NULL THEN NULL
    ELSE REPLACE(FORMAT_NUMBER(valor, 0), ',', '.')
END;

-- 3. Formatação de Percentual (12,34%)
CREATE OR REPLACE FUNCTION mvp.gold.fmt_pct_br(valor DOUBLE, casas INT DEFAULT 2)
RETURNS STRING
LANGUAGE SQL
DETERMINISTIC
CONTAINS SQL
COMMENT 'Converte taxa/percentual para formato brasileiro com símbolo de %'
RETURN CASE 
    WHEN valor IS NULL THEN NULL
    ELSE CONCAT(TRANSLATE(FORMAT_NUMBER(valor, casas), ',.', '.,'), '%')
END;

-- 4. Formatação de Data Brasileira (DD/MM/AAAA)
CREATE OR REPLACE FUNCTION mvp.gold.fmt_data_br(dt DATE)
RETURNS STRING
LANGUAGE SQL
DETERMINISTIC
CONTAINS SQL
COMMENT 'Converte data para formato dd/MM/yyyy'
RETURN CASE 
    WHEN dt IS NULL THEN NULL
    ELSE DATE_FORMAT(dt, 'dd/MM/yyyy')
END;

# Fatos da Camada Gold

In [0]:
%sql
-- ==============================================================================
-- 1. FATO_PIX_TRANSACOES
-- ==============================================================================
CREATE TABLE IF NOT EXISTS mvp.gold.fato_pix_transacoes (
    sk_fato_pix             BIGINT,
    sk_tempo                INT,
    sk_regiao_pagador       BIGINT,
    sk_regiao_recebedor     BIGINT,
    sk_perfil_pagador       BIGINT,
    sk_perfil_recebedor     BIGINT,
    sk_modalidade           BIGINT,
    anomes                  INT,
    data_competencia        DATE,
    quantidade_tx           BIGINT,
    montante_reais          DECIMAL(18, 2),
    ticket_medio_reais      DECIMAL(18, 2),
    _gold_processed_at      TIMESTAMP
)
PARTITIONED BY (anomes)
COMMENT 'Fato agregada de transações Pix conectada ao modelo Star Schema';

MERGE INTO mvp.gold.fato_pix_transacoes AS target
USING (
    SELECT 
        xxhash64(
            anomes, pag_pfpj, rec_pfpj, pag_regiao, rec_regiao,
            pag_faixa_etaria, rec_faixa_etaria, forma_iniciacao, natureza, finalidade
        )                                                                AS sk_fato_pix,
        anomes                                                           AS sk_tempo,
        xxhash64(pag_regiao)                                             AS sk_regiao_pagador,
        xxhash64(rec_regiao)                                             AS sk_regiao_recebedor,
        xxhash64(pag_pfpj, pag_faixa_etaria)                             AS sk_perfil_pagador,
        xxhash64(rec_pfpj, rec_faixa_etaria)                             AS sk_perfil_recebedor,
        xxhash64(forma_iniciacao, natureza, finalidade)                  AS sk_modalidade,
        anomes,
        data_competencia,
        quantidade_tx,
        montante_reais,
        ROUND(montante_reais / NULLIF(quantidade_tx, 0), 2)              AS ticket_medio_reais,
        current_timestamp()                                              AS _gold_processed_at
    FROM mvp.silver.pix_transacoes
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY 
            anomes, pag_pfpj, rec_pfpj, pag_regiao, rec_regiao,
            pag_faixa_etaria, rec_faixa_etaria, forma_iniciacao, natureza, finalidade
        ORDER BY _silver_processed_at DESC
    ) = 1
) AS source
ON target.anomes = source.anomes
   AND target.sk_fato_pix = source.sk_fato_pix
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *;

OPTIMIZE mvp.gold.fato_pix_transacoes
ZORDER BY (sk_regiao_pagador, sk_perfil_pagador, sk_modalidade);

-- ==============================================================================
-- 2. FATO_PIX_FRAUDES_MED
-- ==============================================================================
CREATE TABLE IF NOT EXISTS mvp.gold.fato_pix_fraudes_med (
    sk_tempo                                INT,
    anomes                                  INT,
    data_competencia                        DATE,
    qtde_pix_contestados                    BIGINT,
    qtde_contestacoes_aceitas               BIGINT,
    qtde_contestacoes_rejeitadas            BIGINT,
    taxa_aceite_por_100k                    DECIMAL(12, 4),
    qtde_usuarios_marcados_fraude           BIGINT,
    qtde_chaves_marcadas_fraude             BIGINT,
    valor_contestacoes_aceitas_reais        DECIMAL(18, 2),
    qtde_devolucoes_integrais               BIGINT,
    valor_devolvido_integral_reais          DECIMAL(18, 2),
    qtde_devolucoes_parciais                BIGINT,
    valor_devolvido_parcial_reais           DECIMAL(18, 2),
    valor_residual_nao_devolvido_reais      DECIMAL(18, 2),
    qtde_nao_devolvida_saldo_insuficiente   BIGINT,
    valor_nao_devolvido_saldo_insuficiente  DECIMAL(18, 2),
    qtde_nao_devolvida_conta_encerrada      BIGINT,
    valor_nao_devolvido_conta_encerrada     DECIMAL(18, 2),
    qtde_nao_devolvida_motivos_diversos     BIGINT,
    valor_nao_devolvido_motivos_diversos    DECIMAL(18, 2),
    percentual_devolucao                    DECIMAL(6, 2),
    qtde_bloqueio_cautelar_liberados        BIGINT,
    valor_bloqueio_cautelar_liberado_reais  DECIMAL(18, 2),
    qtde_bloqueio_cautelar_devolvidos       BIGINT,
    valor_bloqueio_cautelar_devolvido_reais DECIMAL(18, 2),
    taxa_efetiva_recuperacao_pct            DECIMAL(10, 2),
    _gold_processed_at                      TIMESTAMP
)
PARTITIONED BY (anomes)
COMMENT 'Fato mensal de fraudes, contestações e devoluções do Mecanismo Especial de Devolução (MED)';

MERGE INTO mvp.gold.fato_pix_fraudes_med AS target
USING (
    SELECT 
        anomes AS sk_tempo,
        anomes,
        data_competencia,
        qtde_pix_contestados,
        qtde_contestacoes_aceitas,
        qtde_contestacoes_rejeitadas,
        taxa_aceite_por_100k,
        qtde_usuarios_marcados_fraude,
        qtde_chaves_marcadas_fraude,
        valor_contestacoes_aceitas_reais,
        qtde_devolucoes_integrais,
        valor_devolvido_integral_reais,
        qtde_devolucoes_parciais,
        valor_devolvido_parcial_reais,
        valor_residual_nao_devolvido_reais,
        qtde_nao_devolvida_saldo_insuficiente,
        valor_nao_devolvido_saldo_insuficiente,
        qtde_nao_devolvida_conta_encerrada,
        valor_nao_devolvido_conta_encerrada,
        qtde_nao_devolvida_motivos_diversos,
        valor_nao_devolvido_motivos_diversos,
        percentual_devolucao,
        qtde_bloqueio_cautelar_liberados,
        valor_bloqueio_cautelar_liberado_reais,
        qtde_bloqueio_cautelar_devolvidos,
        valor_bloqueio_cautelar_devolvido_reais,
        taxa_efetiva_recuperacao_pct,
        current_timestamp() AS _gold_processed_at
    FROM mvp.silver.pix_fraudes_med
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY anomes 
        ORDER BY _silver_processed_at DESC
    ) = 1
) AS source
ON target.anomes = source.anomes
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
52,0,0,52


# Data Mart Analítico

In [0]:
%sql
CREATE OR REPLACE VIEW mvp.gold.dm_pix_painel_executivo AS
WITH resumo_transacoes AS (
    SELECT 
        f.sk_tempo,
        f.anomes,
        f.data_competencia,
        t.ano,
        t.mes,
        t.ano_mes_desc,
        SUM(f.quantidade_tx)                                    AS total_transacoes,
        SUM(f.montante_reais)                                   AS montante_total_reais,
        ROUND(SUM(f.montante_reais) / SUM(f.quantidade_tx), 2)  AS ticket_medio_geral_reais
    FROM mvp.gold.fato_pix_transacoes f
    INNER JOIN mvp.gold.dim_tempo t ON f.sk_tempo = t.sk_tempo
    GROUP BY f.sk_tempo, f.anomes, f.data_competencia, t.ano, t.mes, t.ano_mes_desc
),
populacao_brasil AS (
    SELECT SUM(populacao_recente) AS populacao_total_brasil 
    FROM mvp.gold.dim_regiao 
    WHERE id_regiao_ibge > 0
)
SELECT 
    tx.anomes,
    tx.ano_mes_desc,
    tx.data_competencia,
    
    -- Transacional Pix
    tx.total_transacoes,
    tx.montante_total_reais,
    tx.ticket_medio_geral_reais,
    
    -- Demografia IBGE
    p.populacao_total_brasil,
    ROUND(tx.montante_total_reais / NULLIF(p.populacao_total_brasil, 0), 2) AS montante_per_capita_reais,
    ROUND(tx.total_transacoes / NULLIF(p.populacao_total_brasil, 0), 2)     AS transacoes_per_capita,
    
    -- Segurança e Fraudes MED
    COALESCE(m.qtde_pix_contestados, 0)                     AS qtde_pix_contestados,
    COALESCE(m.valor_contestacoes_aceitas_reais, 0)         AS valor_contestacoes_aceitas_reais,
    COALESCE(m.valor_devolvido_integral_reais, 0)           AS valor_devolvido_integral_reais,
    COALESCE(m.taxa_efetiva_recuperacao_pct, 0.0)           AS taxa_efetiva_recuperacao_pct,
    
    -- Taxa de Contestação Nacional (Contestações sobre o total transacionado no Pix)
    ROUND(
        (COALESCE(m.qtde_pix_contestados, 0) / NULLIF(tx.total_transacoes, 0)) * 100, 
        4
    ) AS taxa_contestacao_pct,
    
    -- Impacto Financeiro da Fraude Aceita sobre o Volume Total
    ROUND(
        (COALESCE(m.valor_contestacoes_aceitas_reais, 0) / NULLIF(tx.montante_total_reais, 0)) * 100, 
        4
    ) AS impacto_fraude_financeira_pct

FROM resumo_transacoes tx
CROSS JOIN populacao_brasil p
LEFT JOIN mvp.gold.fato_pix_fraudes_med m ON tx.sk_tempo = m.sk_tempo
ORDER BY tx.anomes;

# Auditoria e Reconciliação Final (Silver vs. Gold)

In [0]:
%sql
SELECT 
    'fato_pix_transacoes'                                           AS tabela_gold,
    COALESCE(g.anomes, s.anomes)                                    AS anomes,
    COALESCE(g.linhas_gold, 0)                                      AS linhas_gold,
    COALESCE(s.linhas_silver, 0)                                    AS linhas_silver,
    (COALESCE(g.linhas_gold, 0) - COALESCE(s.linhas_silver, 0))     AS dif_linhas,
    
    COALESCE(g.montante_gold, 0.00)                                 AS montante_gold,
    COALESCE(s.montante_silver, 0.00)                               AS montante_silver,
    ROUND(COALESCE(g.montante_gold, 0.00) - COALESCE(s.montante_silver, 0.00), 2) AS dif_montante,
    
    CASE 
        WHEN g.anomes IS NULL 
            THEN '❌ Ausente na Gold (Partição não carregada)'
        WHEN s.anomes IS NULL 
            THEN '❌ Ausente na Silver (Dado órfão na Gold)'
        WHEN g.linhas_gold = s.linhas_silver 
             AND ABS(ROUND(g.montante_gold - s.montante_silver, 2)) = 0.00 
            THEN '✅ 100% Reconciliado'
        ELSE '❌ Divergência Detectada'
    END                                                             AS status_reconciliacao

FROM (
    SELECT 
        anomes, 
        COUNT(1)            AS linhas_gold, 
        SUM(montante_reais) AS montante_gold
    FROM mvp.gold.fato_pix_transacoes
    GROUP BY anomes
) g
FULL OUTER JOIN (
    SELECT 
        anomes, 
        COUNT(1)            AS linhas_silver, 
        SUM(montante_reais) AS montante_silver
    FROM mvp.silver.pix_transacoes
    GROUP BY anomes
) s ON g.anomes = s.anomes
ORDER BY anomes;

tabela_gold,anomes,linhas_gold,linhas_silver,dif_linhas,montante_gold,montante_silver,dif_montante,status_reconciliacao
fato_pix_transacoes,202011,19,19,0,53890784.93,53890784.93,0.00,✅ 100% Reconciliado
fato_pix_transacoes,202012,29,29,0,3293600107.40,3293600107.40,0.00,✅ 100% Reconciliado
fato_pix_transacoes,202101,30,30,0,817058283.67,817058283.67,0.00,✅ 100% Reconciliado
fato_pix_transacoes,202102,35,35,0,361005193.43,361005193.43,0.00,✅ 100% Reconciliado
fato_pix_transacoes,202103,39,39,0,4232818311.21,4232818311.21,0.00,✅ 100% Reconciliado
fato_pix_transacoes,202104,22,22,0,443519465.59,443519465.59,0.00,✅ 100% Reconciliado
fato_pix_transacoes,202105,38,38,0,3478503140.57,3478503140.57,0.00,✅ 100% Reconciliado
fato_pix_transacoes,202106,134,134,0,1879406936.93,1879406936.93,0.00,✅ 100% Reconciliado
fato_pix_transacoes,202107,129,129,0,4966590578.59,4966590578.59,0.00,✅ 100% Reconciliado
fato_pix_transacoes,202108,130,130,0,3218501306.42,3218501306.42,0.00,✅ 100% Reconciliado


# Visões Analíticas

## 1. Visão de Tendência Temporal e Evolução de Risco (Pix vs. Fraudes)



- O que é: Uma visão consolidada mês a mês que acompanha o crescimento do Pix e a segurança das operações.

- Para que serve: Reúne os dados do Pix organizados por mês para alimentar os gráficos do relatório. Com ela, consegue-se acompanhar o ritmo de crescimento dos pagamentos e monitorar, mês a mês, se as contestações de fraude estão aumentando ou sob controle.

In [0]:
%sql
CREATE OR REPLACE VIEW mvp.gold.vw_analise_tendencia_temporal 
COMMENT 'Visão analítica mensal da evolução transacional Pix e indicadores de segurança do MED'
AS
WITH transacoes_mensais AS (
    SELECT 
        sk_tempo,
        SUM(quantidade_tx)                                              AS total_transacoes,
        SUM(montante_reais)                                             AS montante_total_reais,
        CAST(ROUND(SUM(montante_reais) / NULLIF(SUM(quantidade_tx), 0), 2) AS DECIMAL(18, 2)) AS ticket_medio_geral_reais
    FROM mvp.gold.fato_pix_transacoes
    GROUP BY sk_tempo
),
metricas_consolidadas AS (
    SELECT 
        t.sk_tempo,
        t.ano_mes_desc,
        t.data_referencia                                               AS data_competencia,
        t.ano,
        t.mes,
        t.trimestre,
        t.semestre,
        
        -- Volumetria Transacional
        COALESCE(tx.total_transacoes, 0)                                AS total_transacoes,
        COALESCE(tx.montante_total_reais, 0.00)                         AS montante_total_reais,
        COALESCE(tx.ticket_medio_geral_reais, 0.00)                     AS ticket_medio_geral_reais,
        
        -- Indicadores de Risco e Segurança (MED)
        COALESCE(m.qtde_pix_contestados, 0)                             AS qtde_pix_contestados,
        COALESCE(m.qtde_contestacoes_aceitas, 0)                        AS qtde_contestacoes_aceitas,
        COALESCE(m.valor_contestacoes_aceitas_reais, 0.00)              AS valor_contestacoes_aceitas_reais,
        COALESCE(m.valor_devolvido_integral_reais, 0.00)                AS valor_devolvido_integral_reais,
        COALESCE(m.taxa_efetiva_recuperacao_pct, 0.00)                  AS taxa_efetiva_recuperacao_pct,
        
        -- Taxas Relativas
        ROUND(
            (COALESCE(m.qtde_pix_contestados, 0) / NULLIF(tx.total_transacoes, 0)) * 100, 
            4
        )                                                               AS taxa_contestacao_pct,
        ROUND(
            (COALESCE(m.valor_contestacoes_aceitas_reais, 0.00) / NULLIF(tx.montante_total_reais, 0)) * 100, 
            4
        )                                                               AS impacto_fraude_financeira_pct

    FROM mvp.gold.dim_tempo t
    LEFT JOIN transacoes_mensais tx 
        ON t.sk_tempo = tx.sk_tempo
    LEFT JOIN mvp.gold.fato_pix_fraudes_med m 
        ON t.sk_tempo = m.sk_tempo
)
SELECT 
    sk_tempo,
    ano_mes_desc,
    data_competencia,
    ano,
    mes,
    trimestre,
    semestre,
    total_transacoes,
    montante_total_reais,
    ticket_medio_geral_reais,
    qtde_pix_contestados,
    qtde_contestacoes_aceitas,
    valor_contestacoes_aceitas_reais,
    valor_devolvido_integral_reais,
    taxa_efetiva_recuperacao_pct,
    taxa_contestacao_pct,
    impacto_fraude_financeira_pct,
    
    -- Variação Mensal MoM (%)
    ROUND(
        (montante_total_reais - LAG(montante_total_reais, 1) OVER (ORDER BY sk_tempo)) 
        / NULLIF(LAG(montante_total_reais, 1) OVER (ORDER BY sk_tempo), 0) * 100, 
        2
    ) AS variacao_montante_mom_pct,
    ROUND(
        (total_transacoes - LAG(total_transacoes, 1) OVER (ORDER BY sk_tempo)) 
        / NULLIF(LAG(total_transacoes, 1) OVER (ORDER BY sk_tempo), 0) * 100, 
        2
    ) AS variacao_transacoes_mom_pct

FROM metricas_consolidadas;

## 2. Visão de Ranking Regional e Métricas Per Capita

- Nome da consulta: mvp.gold.vw_analise_ranking_regional

- Para que serve: Cruza os dados de pagamentos do Pix com as informações do censo do IBGE. Isso permite alimentar mapas e gráficos que mostram não apenas onde estão os maiores volumes financeiros, mas também o nível de adoção do Pix por pessoa em cada região do Brasil.


In [0]:
%sql
CREATE OR REPLACE VIEW mvp.gold.vw_analise_ranking_regional
COMMENT 'Visão analítica regional agregando volume financeiro e métricas per capita com base no IBGE'
AS
SELECT 
    t.sk_tempo,
    t.ano_mes_desc,
    t.ano,
    t.mes,
    r.id_regiao_ibge,
    r.nome_regiao,
    r.populacao_recente,
    
    -- Volumetria por Região Pagadora
    SUM(f.quantidade_tx)                                                      AS volume_transacoes_pagas,
    SUM(f.montante_reais)                                                     AS montante_total_pago_reais,
    
    -- Ticket Médio Regional
    CAST(ROUND(SUM(f.montante_reais) / NULLIF(SUM(f.quantidade_tx), 0), 2) AS DECIMAL(18, 2)) AS ticket_medio_regiao_reais,
    
    -- Participação no Total Nacional da Competência (% Share)
    CAST(ROUND(
        SUM(f.montante_reais) * 100.0 / 
        NULLIF(SUM(SUM(f.montante_reais)) OVER (PARTITION BY t.sk_tempo), 0), 
        2
    ) AS DECIMAL(5, 2))                                                       AS share_montante_nacional_pct,
    
    -- Intensidade Per Capita (Cruzamento IBGE)
    CAST(ROUND(SUM(f.montante_reais) / NULLIF(r.populacao_recente, 0), 2) AS DECIMAL(18, 2))   AS gasto_per_capita_reais,
    CAST(ROUND(SUM(f.quantidade_tx) * 1.0 / NULLIF(r.populacao_recente, 0), 2) AS DECIMAL(10, 2)) AS tx_por_habitante

FROM mvp.gold.fato_pix_transacoes f
INNER JOIN mvp.gold.dim_tempo t 
    ON f.sk_tempo = t.sk_tempo
INNER JOIN mvp.gold.dim_regiao r 
    ON f.sk_regiao_pagador = r.sk_regiao
WHERE r.id_regiao_ibge > 0 -- Desconsidera registros com região 'NÃO INFORMADO'
GROUP BY 
    t.sk_tempo,
    t.ano_mes_desc,
    t.ano,
    t.mes,
    r.id_regiao_ibge,
    r.nome_regiao,
    r.populacao_recente;

## 3. Visão de Perfil Comportamental e Modalidades Pix

- Nome da consulta: mvp.gold.vw_analise_perfil_risco_operacional

- Para que serve: Organiza as transações combinando o perfil de quem envia e de quem recebe (Pessoa Física ou Jurídica, idade) com o tipo de operação realizada (compra, transferência, chave manual, QR Code). Serve para alimentar gráficos de fluxo, tabelas dinâmicas e mapas de calor que explicam o comportamento dos clientes.

In [0]:
%sql
CREATE OR REPLACE VIEW mvp.gold.vw_analise_perfil_risco_operacional
COMMENT 'Visão analítica de perfil relacional (pagador x recebedor), faixa etária e modalidades de operação'
AS
SELECT 
    t.sk_tempo,
    t.ano_mes_desc,
    t.ano,
    
    -- Perfis de Pagador e Recebedor
    p_pag.tipo_pessoa                                                         AS tipo_pessoa_pagador,
    p_pag.faixa_etaria                                                        AS faixa_etaria_pagador,
    p_pag.ordem_faixa_etaria                                                  AS ordem_faixa_etaria_pagador,
    p_rec.tipo_pessoa                                                         AS tipo_pessoa_recebedor,
    p_rec.faixa_etaria                                                        AS faixa_etaria_recebedor,
    p_rec.ordem_faixa_etaria                                                  AS ordem_faixa_etaria_recebedor,
    CONCAT(p_pag.tipo_pessoa, ' ➔ ', p_rec.tipo_pessoa)                       AS fluxo_relacional_pessoa,
    
    -- Características Operacionais com Semântica Bacen
    m.forma_iniciacao,
    m.forma_iniciacao_desc,
    m.ordem_iniciacao,
    m.natureza,
    m.finalidade,
    
    -- Métricas Agregadas
    SUM(f.quantidade_tx)                                                      AS total_operacoes,
    SUM(f.montante_reais)                                                     AS volume_financeiro_reais,
    CAST(ROUND(SUM(f.montante_reais) / NULLIF(SUM(f.quantidade_tx), 0), 2) AS DECIMAL(18, 2)) AS ticket_medio_reais,
    
    -- Segmentação de Alto Valor (Transações com ticket médio >= R$ 5.000)
    SUM(CASE WHEN f.ticket_medio_reais >= 5000 THEN f.quantidade_tx ELSE 0 END) AS qtd_transacoes_alto_valor,
    SUM(CASE WHEN f.ticket_medio_reais >= 5000 THEN f.montante_reais ELSE 0.00 END) AS volume_alto_valor_reais

FROM mvp.gold.fato_pix_transacoes f
INNER JOIN mvp.gold.dim_tempo t 
    ON f.sk_tempo = t.sk_tempo
INNER JOIN mvp.gold.dim_perfil_cliente p_pag 
    ON f.sk_perfil_pagador = p_pag.sk_perfil
INNER JOIN mvp.gold.dim_perfil_cliente p_rec 
    ON f.sk_perfil_recebedor = p_rec.sk_perfil
INNER JOIN mvp.gold.dim_modalidade_pix m 
    ON f.sk_modalidade = m.sk_modalidade
GROUP BY 
    t.sk_tempo,
    t.ano_mes_desc,
    t.ano,
    p_pag.tipo_pessoa,
    p_pag.faixa_etaria,
    p_pag.ordem_faixa_etaria,
    p_rec.tipo_pessoa,
    p_rec.faixa_etaria,
    p_rec.ordem_faixa_etaria,
    m.forma_iniciacao,
    m.forma_iniciacao_desc,
    m.ordem_iniciacao,
    m.natureza,
    m.finalidade;

In [0]:
%sql
SELECT * FROM mvp.gold.vw_analise_tendencia_temporal ORDER BY sk_tempo DESC LIMIT 5;
SELECT * FROM mvp.gold.vw_analise_ranking_regional ORDER BY sk_tempo DESC, montante_total_pago_reais DESC LIMIT 5;
SELECT * FROM mvp.gold.vw_analise_perfil_risco_operacional ORDER BY volume_financeiro_reais DESC LIMIT 5;

sk_tempo,ano_mes_desc,ano,tipo_pessoa_pagador,faixa_etaria_pagador,ordem_faixa_etaria_pagador,tipo_pessoa_recebedor,faixa_etaria_recebedor,ordem_faixa_etaria_recebedor,fluxo_relacional_pessoa,forma_iniciacao,forma_iniciacao_desc,ordem_iniciacao,natureza,finalidade,total_operacoes,volume_financeiro_reais,ticket_medio_reais,qtd_transacoes_alto_valor,volume_alto_valor_reais
202608,2026/08,2026,PJ,Nao se aplica,9,PJ,Nao se aplica,9,PJ ➔ PJ,MANU,Inserção Manual (Dados de Agência e Conta),2,B2B,PIX,76098049,927361786876.46,12186.41,59884933,879370776068.64
202607,2026/07,2026,PJ,Nao se aplica,9,PJ,Nao se aplica,9,PJ ➔ PJ,MANU,Inserção Manual (Dados de Agência e Conta),2,B2B,PIX,78103550,812531909121.75,10403.26,46971084,741076233157.75
202603,2026/03,2026,PJ,Nao se aplica,9,PJ,Nao se aplica,9,PJ ➔ PJ,MANU,Inserção Manual (Dados de Agência e Conta),2,B2B,PIX,50170739,741079783877.11,14771.16,45688674,728709819605.53
202607,2026/07,2026,PJ,Nao se aplica,9,PJ,Nao se aplica,9,PJ ➔ PJ,DICT,"Chave Pix (Endereçamento via CPF/CNPJ, E-mail, Celular ou EVP)",1,B2B,PIX,51308954,390421814505.54,7609.23,51304681,390415574550.53
202411,2024/11,2024,PJ,Nao se aplica,9,PJ,Nao se aplica,9,PJ ➔ PJ,MANU,Inserção Manual (Dados de Agência e Conta),2,B2B,PIX,24051950,344370112983.88,14317.76,24051950,344370112983.88


# Visões: Código consolidado para Dashboard

In [0]:
%sql
-- ==============================================================================
-- BLOCO 1: VISÃO EXECUTIVA & TENDÊNCIA TEMPORAL
-- ==============================================================================

-- 1.1 Cards de KPIs do Topo (Mês Mais Recente vs. Mês Anterior)
CREATE OR REPLACE VIEW mvp.gold.vw_kpi_topo
COMMENT 'KPIs executivos mais recentes com variação MoM no padrão brasileiro'
AS
SELECT 
    ano_mes_desc                                          AS mes_referencia,
    -- Campos formatados em pt-BR
    mvp.gold.fmt_moeda_br(montante_total_reais)           AS montante_total_br,
    mvp.gold.fmt_pct_br(variacao_montante_mom_pct, 2)     AS variacao_montante_mom_br,
    mvp.gold.fmt_inteiro_br(total_transacoes)             AS total_transacoes_br,
    mvp.gold.fmt_pct_br(variacao_transacoes_mom_pct, 2)   AS variacao_transacoes_mom_br,
    mvp.gold.fmt_moeda_br(ticket_medio_geral_reais)       AS ticket_medio_geral_br,
    mvp.gold.fmt_pct_br(taxa_contestacao_pct, 4)          AS taxa_contestacao_br,
    mvp.gold.fmt_pct_br(taxa_efetiva_recuperacao_pct, 2)   AS taxa_recuperacao_med_br,
    -- Campos numéricos mantidos para compatibilidade com widgets analíticos
    montante_total_reais,
    variacao_montante_mom_pct,
    total_transacoes,
    variacao_transacoes_mom_pct,
    ticket_medio_geral_reais,
    taxa_contestacao_pct,
    taxa_efetiva_recuperacao_pct
FROM mvp.gold.vw_analise_tendencia_temporal
ORDER BY sk_tempo DESC
LIMIT 1;

-- 1.2 Série Temporal Completa para Gráfico de Tendência e Risco
CREATE OR REPLACE VIEW mvp.gold.vw_tendencia_temporal_risco
COMMENT 'Série histórica mensal com métricas e formatação executiva'
AS
SELECT 
    sk_tempo,
    data_competencia,
    mvp.gold.fmt_data_br(data_competencia)                AS data_competencia_br,
    ano_mes_desc,
    -- Métricas Numéricas (para eixos contínuos de gráficos)
    montante_total_reais,
    total_transacoes,
    ticket_medio_geral_reais,
    taxa_contestacao_pct,
    taxa_efetiva_recuperacao_pct,
    variacao_montante_mom_pct,
    -- Métricas Formatadas em pt-BR (para tabelas e tooltips)
    mvp.gold.fmt_moeda_br(montante_total_reais)           AS montante_total_br,
    mvp.gold.fmt_inteiro_br(total_transacoes)             AS total_transacoes_br,
    mvp.gold.fmt_moeda_br(ticket_medio_geral_reais)       AS ticket_medio_br,
    mvp.gold.fmt_pct_br(taxa_contestacao_pct, 4)          AS taxa_contestacao_br,
    mvp.gold.fmt_pct_br(taxa_efetiva_recuperacao_pct, 2)   AS taxa_recuperacao_med_br,
    mvp.gold.fmt_pct_br(variacao_montante_mom_pct, 2)     AS variacao_mom_br
FROM mvp.gold.vw_analise_tendencia_temporal;

-- ==============================================================================
-- BLOCO 2: ANÁLISE REGIONAL & DEMOGRAFIA IBGE
-- ==============================================================================

-- 2.1 Ranking Regional Consolidado e Intensidade Per Capita
CREATE OR REPLACE VIEW mvp.gold.vw_regional_per_capita
COMMENT 'Métricas de adoção normalizadas pela população IBGE com formatação pt-BR'
AS
WITH agregacao_regional AS (
    SELECT 
        nome_regiao,
        populacao_recente,
        SUM(volume_transacoes_pagas)                                            AS total_transacoes_num,
        SUM(montante_total_pago_reais)                                          AS montante_total_pago_num,
        ROUND(SUM(montante_total_pago_reais) / NULLIF(SUM(volume_transacoes_pagas), 0), 2) AS ticket_medio_regiao_num,
        ROUND(
            SUM(montante_total_pago_reais) / NULLIF(SUM(SUM(montante_total_pago_reais)) OVER (), 0) * 100, 
            2
        )                                                                       AS share_nacional_pct_num,
        ROUND(SUM(montante_total_pago_reais) / NULLIF(populacao_recente, 0), 2) AS gasto_per_capita_reais_num,
        ROUND(SUM(volume_transacoes_pagas) * 1.0 / NULLIF(populacao_recente, 0), 2) AS transacoes_por_habitante_num
    FROM mvp.gold.vw_analise_ranking_regional
    GROUP BY nome_regiao, populacao_recente
)
SELECT 
    nome_regiao,
    -- Padrão pt-BR
    mvp.gold.fmt_inteiro_br(populacao_recente)                             AS populacao_br,
    mvp.gold.fmt_inteiro_br(total_transacoes_num)                          AS total_transacoes_br,
    mvp.gold.fmt_moeda_br(montante_total_pago_num)                         AS montante_total_pago_br,
    mvp.gold.fmt_moeda_br(ticket_medio_regiao_num)                         AS ticket_medio_regiao_br,
    mvp.gold.fmt_pct_br(share_nacional_pct_num, 2)                         AS share_nacional_pct_br,
    mvp.gold.fmt_moeda_br(gasto_per_capita_reais_num)                      AS gasto_per_capita_br,
    TRANSLATE(FORMAT_NUMBER(transacoes_por_habitante_num, 2), ',.', '.,')   AS transacoes_por_habitante_br,
    -- Numéricos puros para ordenação/gráficos
    populacao_recente,
    total_transacoes_num                                                   AS total_transacoes,
    montante_total_pago_num                                                AS montante_total_pago,
    ticket_medio_regiao_num                                                AS ticket_medio_regiao,
    share_nacional_pct_num                                                 AS share_nacional_pct,
    gasto_per_capita_reais_num                                             AS gasto_per_capita_reais,
    transacoes_por_habitante_num                                           AS transacoes_por_habitante
FROM agregacao_regional;

-- 2.2 Balança de Fluxos Inter-Regionais (Envio vs. Recebimento)
CREATE OR REPLACE VIEW mvp.gold.vw_balanca_inter_regional
COMMENT 'Saldo líquido de transferência de capital entre macrorregiões'
AS
WITH envios AS (
    SELECT 
        r.nome_regiao,
        SUM(f.montante_reais) AS montante_enviado
    FROM mvp.gold.fato_pix_transacoes f
    INNER JOIN mvp.gold.dim_regiao r ON f.sk_regiao_pagador = r.sk_regiao
    WHERE r.id_regiao_ibge > 0
    GROUP BY r.nome_regiao
),
recebimentos AS (
    SELECT 
        r.nome_regiao,
        SUM(f.montante_reais) AS montante_recebido
    FROM mvp.gold.fato_pix_transacoes f
    INNER JOIN mvp.gold.dim_regiao r ON f.sk_regiao_recebedor = r.sk_regiao
    WHERE r.id_regiao_ibge > 0
    GROUP BY r.nome_regiao
),
consolidado AS (
    SELECT 
        e.nome_regiao,
        e.montante_enviado,
        rc.montante_recebido,
        ROUND(rc.montante_recebido - e.montante_enviado, 2) AS saldo_liquido_regional,
        CASE 
            WHEN rc.montante_recebido >= e.montante_enviado THEN 'Receptora Líquida'
            ELSE 'Emissora Líquida'
        END AS classificacao_fluxo
    FROM envios e
    INNER JOIN recebimentos rc ON e.nome_regiao = rc.nome_regiao
)
SELECT 
    nome_regiao,
    -- Padrão pt-BR
    mvp.gold.fmt_moeda_br(montante_enviado)        AS montante_enviado_br,
    mvp.gold.fmt_moeda_br(montante_recebido)       AS montante_recebido_br,
    mvp.gold.fmt_moeda_br(saldo_liquido_regional)  AS saldo_liquido_regional_br,
    classificacao_fluxo,
    -- Numéricos puros
    montante_enviado,
    montante_recebido,
    saldo_liquido_regional
FROM consolidado;

-- ==============================================================================
-- BLOCO 3: COMPORTAMENTO, FLUXOS PF/PJ & MODALIDADES
-- ==============================================================================

-- 3.1 Matriz Relacional de Pagamento (P2P, P2B, B2P, B2B)
CREATE OR REPLACE VIEW mvp.gold.vw_matriz_relacional_pfpj
COMMENT 'Distribuição relacional com volumes monetários e percentuais formatados'
AS
WITH consolidado_fluxo AS (
    SELECT 
        fluxo_relacional_pessoa,
        CASE 
            WHEN fluxo_relacional_pessoa = 'PF ➔ PF' THEN 'P2P (Transferência Pessoal)'
            WHEN fluxo_relacional_pessoa = 'PF ➔ PJ' THEN 'P2B (Varejo / Comércio)'
            WHEN fluxo_relacional_pessoa = 'PJ ➔ PF' THEN 'B2P (Pagamentos / Salários)'
            WHEN fluxo_relacional_pessoa = 'PJ ➔ PJ' THEN 'B2B (Liquidação Corporativa)'
            ELSE 'Outros'
        END                                                                       AS categoria_fluxo,
        SUM(total_operacoes)                                                      AS total_operacoes_num,
        SUM(volume_financeiro_reais)                                              AS volume_financeiro_num,
        ROUND(
            SUM(volume_financeiro_reais) / NULLIF(SUM(SUM(volume_financeiro_reais)) OVER (), 0) * 100, 
            2
        )                                                                         AS share_volume_pct_num,
        ROUND(SUM(volume_financeiro_reais) / NULLIF(SUM(total_operacoes), 0), 2)  AS ticket_medio_num
    FROM mvp.gold.vw_analise_perfil_risco_operacional
    GROUP BY fluxo_relacional_pessoa
)
SELECT 
    fluxo_relacional_pessoa,
    categoria_fluxo,
    -- Padrão pt-BR
    mvp.gold.fmt_inteiro_br(total_operacoes_num)      AS total_operacoes_br,
    mvp.gold.fmt_moeda_br(volume_financeiro_num)      AS volume_financeiro_br,
    mvp.gold.fmt_pct_br(share_volume_pct_num, 2)      AS share_volume_pct_br,
    mvp.gold.fmt_moeda_br(ticket_medio_num)           AS ticket_medio_br,
    -- Numéricos puros
    total_operacoes_num                               AS total_operacoes,
    volume_financeiro_num                             AS volume_financeiro_reais,
    share_volume_pct_num                              AS share_volume_pct,
    ticket_medio_num                                  AS ticket_medio_reais
FROM consolidado_fluxo;

-- 3.2 Penetração por Canal de Iniciação e Faixa Etária (Incorpora melhorias da Silver)
CREATE OR REPLACE VIEW mvp.gold.vw_canal_vs_faixa_etaria
COMMENT 'Cruzamento de canais Bacen e faixas etárias com ordenação analítica'
AS
WITH metricas_canal AS (
    SELECT 
        faixa_etaria_pagador,
        ordem_faixa_etaria_pagador,
        forma_iniciacao,
        forma_iniciacao_desc,
        ordem_iniciacao,
        SUM(total_operacoes)                                                     AS total_operacoes_num,
        SUM(volume_financeiro_reais)                                             AS volume_financeiro_num,
        ROUND(SUM(volume_financeiro_reais) / NULLIF(SUM(total_operacoes), 0), 2) AS ticket_medio_num
    FROM mvp.gold.vw_analise_perfil_risco_operacional
    GROUP BY 
        faixa_etaria_pagador,
        ordem_faixa_etaria_pagador,
        forma_iniciacao,
        forma_iniciacao_desc,
        ordem_iniciacao
)
SELECT 
    faixa_etaria_pagador,
    ordem_faixa_etaria_pagador,
    forma_iniciacao,
    forma_iniciacao_desc,
    ordem_iniciacao,
    -- Padrão pt-BR
    mvp.gold.fmt_inteiro_br(total_operacoes_num)   AS total_operacoes_br,
    mvp.gold.fmt_moeda_br(volume_financeiro_num)   AS volume_financeiro_br,
    mvp.gold.fmt_moeda_br(ticket_medio_num)        AS ticket_medio_br,
    -- Numéricos puros
    total_operacoes_num                            AS total_operacoes,
    volume_financeiro_num                          AS volume_financeiro_reais,
    ticket_medio_num                               AS ticket_medio_reais
FROM metricas_canal;

-- ==============================================================================
-- BLOCO 4: INTELIGÊNCIA DE SEGURANÇA & EFICÁCIA DO MED
-- ==============================================================================

-- 4.1 Funil Operacional do MED (Contestados -> Aceitos -> Devolvidos)
CREATE OR REPLACE VIEW mvp.gold.vw_funil_med
COMMENT 'Funil de eficiência operacional do MED com percentuais e valores formatados'
AS
WITH funil_calculado AS (
    SELECT 
        m.sk_tempo,
        t.ano_mes_desc,
        m.qtde_pix_contestados,
        m.qtde_contestacoes_aceitas,
        (m.qtde_devolucoes_integrais + m.qtde_devolucoes_parciais)                     AS total_devolucoes_efetuadas,
        ROUND((m.qtde_contestacoes_aceitas * 100.0 / NULLIF(m.qtde_pix_contestados, 0)), 2) AS taxa_aceite_contestacao_pct,
        ROUND(
            ((m.qtde_devolucoes_integrais + m.qtde_devolucoes_parciais) * 100.0 / NULLIF(m.qtde_contestacoes_aceitas, 0)), 
            2
        )                                                                             AS taxa_conclusao_devolucao_pct,
        m.valor_contestacoes_aceitas_reais,
        (m.valor_devolvido_integral_reais + m.valor_devolvido_parcial_reais)           AS valor_total_recuperado_reais,
        m.taxa_efetiva_recuperacao_pct
    FROM mvp.gold.fato_pix_fraudes_med m
    INNER JOIN mvp.gold.dim_tempo t ON m.sk_tempo = t.sk_tempo
)
SELECT 
    sk_tempo,
    ano_mes_desc,
    -- Padrão pt-BR
    mvp.gold.fmt_inteiro_br(qtde_pix_contestados)                AS qtde_pix_contestados_br,
    mvp.gold.fmt_inteiro_br(qtde_contestacoes_aceitas)           AS qtde_contestacoes_aceitas_br,
    mvp.gold.fmt_inteiro_br(total_devolucoes_efetuadas)          AS total_devolucoes_efetuadas_br,
    mvp.gold.fmt_pct_br(taxa_aceite_contestacao_pct, 2)          AS taxa_aceite_contestacao_br,
    mvp.gold.fmt_pct_br(taxa_conclusao_devolucao_pct, 2)         AS taxa_conclusao_devolucao_br,
    mvp.gold.fmt_moeda_br(valor_contestacoes_aceitas_reais)      AS valor_contestacoes_aceitas_br,
    mvp.gold.fmt_moeda_br(valor_total_recuperado_reais)          AS valor_total_recuperado_br,
    mvp.gold.fmt_pct_br(taxa_efetiva_recuperacao_pct, 2)         AS taxa_efetiva_recuperacao_med_br,
    -- Numéricos puros
    qtde_pix_contestados,
    qtde_contestacoes_aceitas,
    total_devolucoes_efetuadas,
    taxa_aceite_contestacao_pct,
    taxa_conclusao_devolucao_pct,
    valor_contestacoes_aceitas_reais,
    valor_total_recuperado_reais,
    taxa_efetiva_recuperacao_pct
FROM funil_calculado;

-- 4.2 Diagnóstico dos Gargalos de Insucesso na Devolução
CREATE OR REPLACE VIEW mvp.gold.vw_gargalos_devolucao_med
COMMENT 'Diagnóstico dos gargalos de devolução de fraudes com moeda e taxas pt-BR'
AS
WITH gargalos_calculados AS (
    SELECT 
        m.sk_tempo,
        t.ano_mes_desc,
        m.valor_contestacoes_aceitas_reais,
        (m.valor_devolvido_integral_reais + m.valor_devolvido_parcial_reais)           AS valor_recuperado,
        m.valor_nao_devolvido_saldo_insuficiente                                       AS valor_perdido_saldo_insuficiente,
        m.valor_nao_devolvido_conta_encerrada                                          AS valor_perdido_conta_encerrada,
        m.valor_nao_devolvido_motivos_diversos                                         AS valor_perdido_outros_motivos,
        ROUND(
            (m.valor_nao_devolvido_saldo_insuficiente * 100.0 / NULLIF(m.valor_contestacoes_aceitas_reais, 0)), 
            2
        )                                                                              AS impacto_saldo_insuficiente_pct,
        ROUND(
            (m.valor_nao_devolvido_conta_encerrada * 100.0 / NULLIF(m.valor_contestacoes_aceitas_reais, 0)), 
            2
        )                                                                              AS impacto_conta_encerrada_pct
    FROM mvp.gold.fato_pix_fraudes_med m
    INNER JOIN mvp.gold.dim_tempo t ON m.sk_tempo = t.sk_tempo
)
SELECT 
    sk_tempo,
    ano_mes_desc,
    -- Padrão pt-BR
    mvp.gold.fmt_moeda_br(valor_contestacoes_aceitas_reais)        AS valor_contestacoes_aceitas_br,
    mvp.gold.fmt_moeda_br(valor_recuperado)                        AS valor_recuperado_br,
    mvp.gold.fmt_moeda_br(valor_perdido_saldo_insuficiente)        AS valor_perdido_saldo_insuficiente_br,
    mvp.gold.fmt_moeda_br(valor_perdido_conta_encerrada)          AS valor_perdido_conta_encerrada_br,
    mvp.gold.fmt_moeda_br(valor_perdido_outros_motivos)           AS valor_perdido_outros_motivos_br,
    mvp.gold.fmt_pct_br(impacto_saldo_insuficiente_pct, 2)         AS impacto_saldo_insuficiente_br,
    mvp.gold.fmt_pct_br(impacto_conta_encerrada_pct, 2)           AS impacto_conta_encerrada_br,
    -- Numéricos puros
    valor_contestacoes_aceitas_reais,
    valor_recuperado,
    valor_perdido_saldo_insuficiente,
    valor_perdido_conta_encerrada,
    valor_perdido_outros_motivos,
    impacto_saldo_insuficiente_pct,
    impacto_conta_encerrada_pct
FROM gargalos_calculados;